# 03 · Jubilación mínima, bono y costo fiscal

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/santiagoriverti/IPC_jubilados/blob/main/notebooks/03_haberes_y_fiscal.ipynb)

- Haber mínimo (datos.gob.ar) y bono (decretos; congelado en $70.000 desde marzo 2024).
- Movilidad vigente (DNU 274/2024): el haber de cada mes sube lo que subió el IPC dos meses antes.
- Contrafactual: la misma regla con el IIJP. Costo fiscal con el gasto real en jubilaciones y pensiones
  (IMIG, repo `cuentas_publicas`).
- Al final se descarga un ZIP con el Excel de resultados y los gráficos.

In [ ]:
# Configuracion: en Colab clona (o actualiza) el repo; en local usa la carpeta del repo
import os, sys, subprocess
REPO = 'https://github.com/santiagoriverti/IPC_jubilados.git'
if 'google.colab' in sys.modules:
    RAIZ = '/content/IPC_jubilados'
    if os.path.exists(RAIZ):
        subprocess.run(['git', '-C', RAIZ, 'pull', '-q'], check=True)
    else:
        subprocess.run(['git', 'clone', '-q', '--depth', '1', REPO, RAIZ], check=True)
else:  # subir desde la carpeta actual hasta encontrar el repo
    RAIZ = os.getcwd()
    while not os.path.exists(os.path.join(RAIZ, 'src', 'pipeline.py')) and os.path.dirname(RAIZ) != RAIZ:
        RAIZ = os.path.dirname(RAIZ)
sys.path.insert(0, RAIZ)

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from src import pipeline, graficos, indices, ponderaciones as pond
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
pd.set_option('display.max_columns', 30)

In [ ]:
N_BOOTSTRAP = 0   # este notebook no usa los intervalos; poner 500 si se quiere el Excel completo
R = pipeline.calcular(n_bootstrap=N_BOOTSTRAP)
hab = R['haberes']

## La regla de movilidad se cumple

Desde mayo 2024 el aumento del haber mínimo coincide con el IPC de dos meses antes (1 decimal).

In [ ]:
r = R['regla_movilidad'].copy()
r[['aumento_observado', 'ipc_t_menos_2']] *= 100   # en %
display(r.tail(8))
print('Meses que cumplen la regla:', r['ok'].sum(), 'de', len(r))

## Poder adquisitivo (noviembre 2023 = 100)

El haber mínimo **sin bono** está ~10% por encima de noviembre 2023 en términos reales; **con el bono**
está ~10% por debajo. La pérdida del jubilado de la mínima viene sobre todo del bono congelado, no del
índice con que se actualiza el haber. (El TP calculaba −9,8% dividiendo el haber de diciembre por la
inflación que incluía diciembre: mezclaba meses.)

In [ ]:
fig = graficos.g05_haber_real(R); plt.show()

In [ ]:
b = pd.Period('2023-11', 'M')
u = hab.dropna(subset=['real_haber_ipc']).index.max()
t = pd.DataFrame({
    'sin bono, IPC': hab['real_haber_ipc'] / hab.loc[b, 'real_haber_ipc'] - 1,
    'sin bono, IIJP': hab['real_haber_iijp'] / hab.loc[b, 'real_haber_iijp'] - 1,
    'con bono, IPC': hab['real_con_bono_ipc'] / hab.loc[b, 'real_con_bono_ipc'] - 1,
    'con bono, IIJP': hab['real_con_bono_iijp'] / hab.loc[b, 'real_con_bono_iijp'] - 1,
}).loc[['2023-12', '2024-02', '2024-12', '2025-12', str(u)]] * 100
display(t.round(1))
print(f"Bono de $70.000 actualizado por IPC desde marzo 2024 en {u}: ${hab.loc[u, 'bono_indexado_ipc']:,.0f}")

## Contrafactual: haber indexado por IIJP desde abril 2024

In [ ]:
c = hab.loc['2024-04':, ['haber_minimo', 'haber_cf_ipc', 'haber_cf_iijp']].copy()
c['diferencia_%'] = (hab['factor_iijp'] - 1) * 100
display(c.iloc[::3])
u2 = hab['factor_iijp'].dropna().index.max()
print(f"En {u2} el haber indexado por IIJP seria {100 * (hab.loc[u2, 'factor_iijp'] - 1):.2f}% mayor: "
      f"${hab.loc[u2, 'haber_cf_iijp'] - hab.loc[u2, 'haber_cf_ipc']:,.0f} por mes en la minima.")

## Costo fiscal

Como la movilidad aplica el mismo porcentaje a todos los haberes, el costo extra de cada mes es el
gasto en jubilaciones y pensiones (contributivas + no contributivas, IMIG) por (factor − 1). Es una cota
superior: ese gasto incluye el bono, que no se indexa.

In [ ]:
fig = graficos.g06_costo_fiscal(R); plt.show()
fa = R['fiscal_anual']
display(fa[['meses', 'gasto_previsional', 'costo_extra', 'costo_pct_gasto_prev', 'costo_pct_res_financiero',
            'costo_pct_pib']])

## ¿Y si la brecha fuera permanente?

Si el IIJP superara al IPC en *d* puntos por año, el costo relativo al gasto previsional sería
(1 + d)ⁿ − 1, **independiente de la demografía** (el envejecimiento escala igual el gasto con y sin el
cambio de índice). En % del PIB se usa la relación gasto previsional / PIB del último año completo.

In [ ]:
print(f"Gasto previsional / PIB {R['meta']['anio_pib_ref']}: {R['meta']['gasto_prev_pct_pib']:.2f}%")
display(R['fiscal_desvio_permanente'].pivot(index='desvio_pp_anual', columns='anios', values='costo_pct_pib').round(2))

## Descargar resultados (Excel + gráficos)

In [ ]:
import shutil
sys.path.insert(0, os.path.join(RAIZ, 'scripts'))
from construir import exportar
exportar(R)
zip_path = shutil.make_archive(os.path.join(RAIZ, 'IIJP_resultados'), 'zip', os.path.join(RAIZ, 'output'))
print('ZIP:', zip_path)
if 'google.colab' in sys.modules:
    from google.colab import files
    files.download(zip_path)